# Crate architecture and the pipeline

`onceover` is a Cargo workspace of six crates with one direction of
dependencies, plus a thin Python package bound by
[`pyo3`](https://github.com/PyO3/pyo3) and built with maturin. Every
algorithm is Rust; Python is a boundary, not an engine.

```
onceover-core      typed column extraction from polars, exact value
                   counting, column cards, duplicates, date utilities
onceover-deps      keys and exact relationships: FD/AFD discovery,
                   unique column combinations, sortedness
onceover-mines     the model and statistical passes: encoding,
                   predictability screen (histogram GBM), decision
                   tables, PRISM, dependence measures, arithmetic
                   identities, local linear regimes, nominal
                   associations, date/sequence intelligence
onceover-report    finding types, evidence, consolidation and selection,
                   report serialization (no polars dependency)
onceover           the pipeline: budget tiers, detector registry,
                   orchestration, coverage
onceover-py        the pyo3 bindings: profile(df) and the report object
```

`onceover-report` depends on neither polars nor the mining crates: the
statement schema is the stable center, and everything else writes *into*
it. That is what keeps the public taxonomy independent of which detector
happens to run.

## The pipeline

One `profile` call runs in a fixed order — cheap exact passes first,
their outputs feeding the expensive ones:

1. **Extraction** — the polars `DataFrame` becomes a `TypedTable` of
   compact typed columns (zero-copy where the dtype allows it).
2. **Column cards** — exact per-column statistics, roles, concentration,
   format patterns; the card findings are emitted here.
3. **Row duplicates** — exact, with the logical shortcut when a proven
   key makes duplicates impossible.
4. **Keys** — single-column uniqueness from the cards' exact counts,
   composite-key screening and verification.
5. **Functional dependencies** — screen, lattice, verify; approximate
   dependencies and suppression.
6. **Sortedness and dates** — exact scans.
7. **Encoding** — the shared matrix for every model pass.
8. **Predictability screen** — every eligible column as a target.
9. **Decision tables, PRISM, regimes** — the model follow-ups, reading
   the screen's shortlists and fit in that order.
10. **Nominal associations** — pairwise value implications over the
    encoded matrix.
11. **Dependence and identities** — pairwise statistics, then affine
    probes with full-row verification.
12. **Rule evaluation** — the shared predicate evaluator over every
    conditional-rule candidate.
13. **Consolidation and selection** — equivalent statements merge,
    report-wide limits apply, ids are hashed from the canonical
    statement.

Between passes a deadline check (when one is set) records skipped work
in coverage — nothing is interrupted mid-computation.

## The detector registry

Detectors are registered separately from the taxonomy: each declares an
id, the categories it can produce, and the detectors it requires as
resources. Selecting `["prism", "association_rules"]` runs those two and
whatever they need (both require the predictability screen's encoding
and targets), and nothing else contributes findings. Required detectors
may execute to supply shared state without emitting findings of their
own; an empty selection still returns cards and coverage.

In [ ]:
import json

import onceover

for d in onceover.available_detectors():
    print(f"{d['id']:24s} -> {d['categories']}"
          + (f"   requires: {d['requires']}" if d["requires"] else ""))

Rust extensions implement the same contract: a `Detector` trait with a
`DetectorDescriptor` and a `discover(&DetectorContext)` returning
finding candidates. The pipeline validates their output categories,
subjects, and statement definitions; invalid results are reported in
coverage rather than silently reshaping the report.

## Budget tiers

The tiers configure every pass at once. The exact numbers behind the
table live in one place (`TierConfig`), and the coverage block quotes
them back on every run:

| | `quick` | `standard` | `deep` |
|---|---|---|---|
| Exact passes (cards, duplicates, keys, sortedness, dates) | full | full | full |
| Composite keys | — | pairs, verify ≤ 16 | pairs + triples, verify ≤ 24 |
| FD: max LHS size / verification rows | — | 2 / 200,000 | 3 / 500,000 |
| Encoded rows / features | 10,000 / 32 | 20,000 / 48 | 50,000 / 64 |
| GBM fitting rows / rounds | — | 5,000 / 30 | 10,000 / 40 |
| Decision-table / PRISM targets | — | 6 / 6 | 10 / 10 |
| Regimes: targets / rows / depth | — | 4 / 5,000 / 3 | 8 / 10,000 / 4 |
| Dependence pairs (capped) | — | 496 | 2,016 |
| Identity candidate pairs | — | 100 | 200 |
| Association findings | — | 20 | 40 |

`quick` is the integrity tier: cards, duplicates, single-column keys,
sortedness, dates — everything exact, no relationship models. The
`sample_rows`, `max_predictors`, and `max_pairs` overrides rescale the
sampled work without touching the exact passes
([](../usage/coverage.ipynb)).